# AWS Lab 2 | Serverless file processing along with S3 & Lambda

Jesse Pinkman Inc. receives its financial report files from its offices around the nation daily. Instead of using the prior system which is running a server around the clock to watch for new files they wanted to implement a pipeline that triggers automatically once on upload then processes the file & moves it to into an archive bucket.

## Initial setup

In [ ]:
!pip install boto3 --quiet
!aws sts get-caller-identity

setting up the variables.

In [ ]:
import boto3, json, subprocess, zipfile, os, time

REGION     = "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

INCOMING_BUCKET  = f"lab2-incoming-{ACCOUNT_ID}"
PROCESSED_BUCKET = f"lab2-processed-{ACCOUNT_ID}"
FUNCTION_NAME    = "lab2-file-processor"
ROLE_NAME        = "lab2-lambda-role"

print(f"Account : {ACCOUNT_ID}")
print(f"Incoming: {INCOMING_BUCKET}")
print(f"Archive : {PROCESSED_BUCKET}")

## Creating the S3 buckets (two)

first one is for the incoming files & the second is one for the processed files. 

In [ ]:
s3 = boto3.client("s3", region_name=REGION)

for bucket in [INCOMING_BUCKET, PROCESSED_BUCKET]:
    s3.create_bucket(Bucket=bucket)
    s3.put_public_access_block(
        Bucket=bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True,
            "IgnorePublicAcls": True,
            "BlockPublicPolicy": True,
            "RestrictPublicBuckets": True
        }
    )
    print(f"created: {bucket}")

## Creating the IAM role

Lambda has to be assigned a role in order to read & write S3 in order to publish the CloudWatch logs. 

In [ ]:
iam = boto3.client("iam")

trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

role = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))
ROLE_ARN = role["Role"]["Arn"]

for policy in [
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
]:
    iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy)

print(f"role: {ROLE_ARN}")
time.sleep(10)

## Lambda function

Zip and deploy lambda function.

each S3 event is read, it logs the file metadata & copies the file to the archive bucket then deletes the original.

In [ ]:
PROCESSED_BUCKET_REF = PROCESSED_BUCKET

code = f'''
import boto3, json, urllib.parse
from datetime import datetime

s3 = boto3.client("s3")
DEST = "{PROCESSED_BUCKET_REF}"

def lambda_handler(event, context):
    for record in event["Records"]:
        src    = record["s3"]["bucket"]["name"]
        key    = urllib.parse.unquote_plus(record["s3"]["object"]["key"])
        size   = record["s3"]["object"]["size"]
        ext    = key.rsplit(".", 1)[-1].lower() if "." in key else "unknown"

        print(json.dumps({{
            "file": key,
            "size_kb": round(size / 1024, 2),
            "type": ext,
            "processed_at": datetime.utcnow().isoformat()
        }}))

        s3.copy_object(Bucket=DEST, CopySource={{"Bucket": src, "Key": key}}, Key=key)
        s3.delete_object(Bucket=src, Key=key)
        print(f"moved {{key}} to {{DEST}}")
'''

os.makedirs("fn", exist_ok=True)
with open("fn/lambda_function.py", "w") as f:
    f.write(code)

with zipfile.ZipFile("fn.zip", "w") as z:
    z.write("fn/lambda_function.py", "lambda_function.py")

print("packaged fn.zip")

deploying the function to lambda.

In [ ]:
lam = boto3.client("lambda", region_name=REGION)

with open("fn.zip", "rb") as f:
    lam.create_function(
        FunctionName = FUNCTION_NAME,
        Runtime      = "python3.12",
        Role         = ROLE_ARN,
        Handler      = "lambda_function.lambda_handler",
        Code         = {"ZipFile": f.read()},
        Timeout      = 30
    )

FUNCTION_ARN = lam.get_function(FunctionName=FUNCTION_NAME)["Configuration"]["FunctionArn"]
print(f"deployed: {FUNCTION_ARN}")

the S3 is given permission to invoke lambda & attach the trigger to the incoming bucket.

In [ ]:
lam.add_permission(
    FunctionName  = FUNCTION_NAME,
    StatementId   = "s3-invoke",
    Action        = "lambda:InvokeFunction",
    Principal     = "s3.amazonaws.com",
    SourceArn     = f"arn:aws:s3:::{INCOMING_BUCKET}"
)

s3.put_bucket_notification_configuration(
    Bucket=INCOMING_BUCKET,
    NotificationConfiguration={
        "LambdaFunctionConfigurations": [{
            "LambdaFunctionArn": FUNCTION_ARN,
            "Events": ["s3:ObjectCreated:*"]
        }]
    }
)

print("trigger attached")

## Testing

uploaded sample/test files to the incoming bucket so that I could verify that Lambda moves them to the archive.

In [ ]:
test_files = {
    "q2_report.csv":    "client_id,value\nC001,1500000\nC002,980000",
    "trades_may.json":  json.dumps({"month": "2026-05", "total": 847}),
    "memo.txt":         "Q2 reports cleared for archival."
}

for name, content in test_files.items():
    s3.put_object(Bucket=INCOMING_BUCKET, Key=name, Body=content)
    print(f"uploaded: {name}")

time.sleep(6)

double check that the incoming bucket is empty and the three files are in the processed bucket.

In [ ]:
incoming  = s3.list_objects_v2(Bucket=INCOMING_BUCKET).get("Contents", [])
processed = s3.list_objects_v2(Bucket=PROCESSED_BUCKET).get("Contents", [])

print(f"incoming  (expect 0): {len(incoming)}")
print(f"processed (expect 3): {len(processed)}")
for obj in processed:
    print(f"  {obj['Key']}")

## Cloudwatch Logs

each invocation in lambda writes it to a log group & pulls the most recent log events to see the processing records.

In [ ]:
logs      = boto3.client("logs", region_name=REGION)
LOG_GROUP = f"/aws/lambda/{FUNCTION_NAME}"

streams = logs.describe_log_streams(
    logGroupName=LOG_GROUP,
    orderBy="LastEventTime",
    descending=True,
    limit=1
)["logStreams"]

events = logs.get_log_events(
    logGroupName=LOG_GROUP,
    logStreamName=streams[0]["logStreamName"]
)["events"]

for e in events:
    msg = e["message"].strip()
    if msg:
        print(msg)

## Cleanup

deleting lambda & detaching the policies then deleting the roles.

In [ ]:
lam.delete_function(FunctionName=FUNCTION_NAME)

# Empty and delete both buckets
for bucket in [INCOMING_BUCKET, PROCESSED_BUCKET]:
    objs = s3.list_objects_v2(Bucket=bucket).get("Contents", [])
    for obj in objs:
        s3.delete_object(Bucket=bucket, Key=obj["Key"])
    s3.delete_bucket(Bucket=bucket)
    print(f"deleted: {bucket}")

for policy in [
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
]:
    iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy)
iam.delete_role(RoleName=ROLE_NAME)

print("done")